# Behavioural cleaning and feature engineering after construction

After Chapters 21–24 implement point-in-time joins, cleaning and features visibly, this integration lab calls the promoted package and checks that the original synthetic outcome remains rational.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
from creditriskbook.data import make_behavioral_credit_history
from creditriskbook.data.cleaning import clean_monthly_performance
from creditriskbook.features import build_behavioral_features

case = make_behavioral_credit_history(n_customers=200, months=18, seed=2401)
refs = case.applications[["customer_id", "reference_date"]]
cleaning = clean_monthly_performance(case.monthly_performance, refs)
features = build_behavioral_features(
    cleaning.clean, case.contracts, refs, enquiries=case.bureau_enquiries
)
model_table = case.applications.merge(
    features, on=["customer_id", "reference_date"], validate="one_to_one"
)
print("model table:", model_table.shape)
print("cleaning issues:", len(cleaning.issues))
assert cleaning.issues.empty

In [ ]:
selected = [
    "max_dpd_6m", "last_dpd", "count_dpd30_6m",
    "count_contracts_last_6m", "current_utilisation",
]
print(model_table[selected].head().round(4).to_string(index=False))

In [ ]:
import pandas as pd

bands = pd.qcut(model_table["max_dpd_6m"], q=4, duplicates="drop")
characteristic = (
    model_table.assign(band=bands)
    .groupby("band", observed=True)["default_12m"]
    .agg(observations="size", defaults="sum", default_rate="mean")
)
print(characteristic.round(4))
assert characteristic["default_rate"].is_monotonic_increasing

The increasing rate across `max_dpd_6m` bands is a generator rationality check, not evidence that real data must be forced to monotonicity. Real portfolios require source, cohort, policy, uncertainty and stability analysis.